# Lyric-Based Music Genre Classification
**STAT 359 Final Project — Colab (GPU-only tasks)**

This notebook contains only tasks that require GPU:
- Data preparation (downloads from HuggingFace)
- Model training (Phase 1 + Phase 2)
- Evaluation
- Embedding extraction + visualization
- Attention analysis
- Feature attribution

Non-GPU analysis (genre similarity, LLM baseline) runs locally.

## 0. Setup & Install Dependencies

In [ ]:
!pip install -q --no-deps peft captum
!pip install -q transformers datasets accelerate umap-learn pyyaml

In [ ]:
import os, re, json, random
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import f1_score, accuracy_score, classification_report, confusion_matrix
from sklearn.manifold import TSNE
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm

from datasets import load_dataset, DatasetDict, ClassLabel, load_from_disk, concatenate_datasets
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    GPT2ForSequenceClassification, T5ForConditionalGeneration,
    get_cosine_schedule_with_warmup,
)
from peft import LoraConfig, get_peft_model, PeftModel, TaskType

try:
    import umap
    HAS_UMAP = True
except ImportError:
    HAS_UMAP = False

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

sns.set_theme(style="whitegrid", font_scale=1.1)

In [ ]:
# Create output directories
for d in ["data/processed", "data/processed_phase2",
          "results/figures/attention", "results/figures/attribution",
          "results/checkpoints",
          "results/figures_optimized/attention", "results/figures_optimized/attribution",
          "results/checkpoints_optimized"]:
    os.makedirs(d, exist_ok=True)

## 1. Configuration

In [ ]:
# Phase 1: Architecture comparison (all 3 models, same hyperparameters)
CFG = {
    "data": {
        "dataset_name": "sebastiandizon/genius-song-lyrics",
        "genres": ["rap", "pop", "rock", "country", "rb"],
        "samples_per_genre": 2000,
        "samples_per_genre_override": {"pop": 2500},
        "max_chars": 2000,
        "train_ratio": 0.8,
        "val_ratio": 0.1,
        "test_ratio": 0.1,
        "seed": 42,
    },
    "training": {
        "max_seq_len": 512,
        "batch_size": 64,
        "learning_rate": 2e-4,
        "weight_decay": 0.01,
        "epochs": 10,
        "warmup_ratio": 0.1,
        "early_stopping_patience": 3,
        "gradient_accumulation_steps": 1,
        "fp16": False,
        "label_smoothing": 0.0,
        "augment": False,
    },
    "lora": {
        "rank": 16,
        "alpha": 32,
        "dropout": 0.1,
    },
}

LABEL_MAP = {"rap": 0, "pop": 1, "rock": 2, "country": 3, "rb": 4}
LABEL_NAMES = {0: "rap", 1: "pop", 2: "rock", 3: "country", 4: "r&b"}
GENRE_DISPLAY = {0: "Rap/Hip-Hop", 1: "Pop", 2: "Rock", 3: "Country", 4: "R&B"}
NUM_LABELS = 5

DATA_DIR = "data/processed"
FIG_DIR = "results/figures"
CKPT_DIR = "results/checkpoints"

## 2. Data Preparation

In [ ]:
def clean_lyrics(text, max_chars=2000):
    text = re.sub(r"\[.*?\]", "", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    text = text.strip()
    return text[:max_chars] if len(text) > max_chars else text


def prepare_dataset(data_cfg, label_map, output_dir):
    genres = data_cfg["genres"]

    print("Loading dataset ...")
    ds = load_dataset(data_cfg["dataset_name"], split="train")
    print(f"  Total rows: {len(ds):,}")

    print("Filtering to English-only lyrics ...")
    ds = ds.filter(lambda r: r.get("language_cld3") == "en" and r.get("language_ft") == "en", num_proc=2)
    print(f"  After English filter: {len(ds):,}")

    genre_set = set(genres)
    ds = ds.filter(lambda r: r["tag"] in genre_set, num_proc=2)
    print(f"  After genre filter: {len(ds):,}")

    rng = np.random.default_rng(data_cfg["seed"])
    tags = np.array(ds["tag"])
    override = data_cfg.get("samples_per_genre_override") or {}
    default_n = data_cfg["samples_per_genre"]
    sampled_indices = []
    for genre in genres:
        idxs = np.where(tags == genre)[0]
        target = override.get(genre, default_n)
        n = min(target, len(idxs))
        chosen = rng.choice(idxs, size=n, replace=False)
        sampled_indices.extend(chosen.tolist())
        print(f"  {genre}: sampled {n} from {len(idxs):,}")

    rng.shuffle(sampled_indices)
    ds = ds.select(sampled_indices)
    print(f"  Total sampled: {len(ds):,}")

    ds = ds.map(lambda r: {
        "lyrics": clean_lyrics(r["lyrics"], data_cfg["max_chars"]),
        "label": label_map[r["tag"]],
        "genre": r["tag"],
    }, remove_columns=[c for c in ds.column_names if c not in ("lyrics", "label", "genre", "title", "artist")])

    ds = ds.filter(lambda r: len(r["lyrics"].split()) >= 20)
    print(f"  After cleaning: {len(ds):,}")

    genre_names_sorted = [g for g, _ in sorted(label_map.items(), key=lambda x: x[1])]
    ds = ds.cast_column("label", ClassLabel(names=genre_names_sorted))

    seed = data_cfg["seed"]
    split1 = ds.train_test_split(test_size=data_cfg["test_ratio"], seed=seed, stratify_by_column="label")
    split2 = split1["train"].train_test_split(
        test_size=data_cfg["val_ratio"] / (1 - data_cfg["test_ratio"]), seed=seed, stratify_by_column="label"
    )
    dataset = DatasetDict({"train": split2["train"], "validation": split2["test"], "test": split1["test"]})

    def balance_split(split_ds, n_classes, seed):
        rng = np.random.default_rng(seed)
        labels = np.array(split_ds["label"])
        n_per_class = min(int((labels == c).sum()) for c in range(n_classes))
        selected = []
        for c in range(n_classes):
            idx = np.where(labels == c)[0]
            selected.extend(rng.choice(idx, size=n_per_class, replace=False).tolist())
        rng.shuffle(selected)
        return split_ds.select(selected)

    dataset["validation"] = balance_split(dataset["validation"], len(genres), seed)
    dataset["test"] = balance_split(dataset["test"], len(genres), seed)
    dataset.save_to_disk(output_dir)

    for name, split in dataset.items():
        g = np.array(split["genre"])
        counts = {genre: int((g == genre).sum()) for genre in genres}
        print(f"  {name}: {len(split)} rows  {counts}")
    return dataset


# Prepare Phase 1 dataset
dataset = prepare_dataset(CFG["data"], LABEL_MAP, DATA_DIR)

## 3. Dataset Classes

In [ ]:
class LyricsDataset(Dataset):
    def __init__(self, split, tokenizer, max_length=512, data_dir=DATA_DIR, augment=False):
        ds = load_from_disk(data_dir)
        self.data = ds[split]
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.augment = augment

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data[idx]
        text = row["lyrics"]
        if self.augment:
            words = text.split()
            if len(words) > 100:
                keep = random.randint(len(words) // 2, len(words))
                start = random.randint(0, len(words) - keep)
                text = " ".join(words[start:start + keep])
        enc = self.tokenizer(text, truncation=True, max_length=self.max_length,
                             padding="max_length", return_tensors="pt")
        item = {k: v.squeeze(0) for k, v in enc.items()}
        item["labels"] = torch.tensor(row["label"], dtype=torch.long)
        return item


class T5LyricsDataset(Dataset):
    def __init__(self, split, tokenizer, max_length=512, data_dir=DATA_DIR, target_max_length=8):
        ds = load_from_disk(data_dir)
        self.data = ds[split]
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.target_max_length = target_max_length

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data[idx]
        source = self.tokenizer("classify genre: " + row["lyrics"], truncation=True,
                                max_length=self.max_length, padding="max_length", return_tensors="pt")
        target = self.tokenizer(LABEL_NAMES[row["label"]], truncation=True,
                                max_length=self.target_max_length, padding="max_length", return_tensors="pt")
        target_ids = target["input_ids"].squeeze(0)
        target_ids[target_ids == self.tokenizer.pad_token_id] = -100
        item = {k: v.squeeze(0) for k, v in source.items()}
        item["labels"] = target_ids
        item["genre_label"] = torch.tensor(row["label"], dtype=torch.long)
        return item

## 4. Model Builders

In [ ]:
def build_roberta(lora_rank, lora_alpha, lora_dropout, target_modules=None):
    if target_modules is None:
        target_modules = ["query", "value"]
    tokenizer = AutoTokenizer.from_pretrained("roberta-base")
    model = AutoModelForSequenceClassification.from_pretrained("roberta-base", num_labels=NUM_LABELS)
    lora_cfg = LoraConfig(task_type=TaskType.SEQ_CLS, r=lora_rank,
                          lora_alpha=lora_alpha, lora_dropout=lora_dropout,
                          target_modules=target_modules)
    model = get_peft_model(model, lora_cfg)
    model.print_trainable_parameters()
    return model, tokenizer


def build_gpt2(lora_rank, lora_alpha, lora_dropout):
    tokenizer = AutoTokenizer.from_pretrained("gpt2")
    tokenizer.pad_token = tokenizer.eos_token
    model = GPT2ForSequenceClassification.from_pretrained("gpt2", num_labels=NUM_LABELS)
    model.config.pad_token_id = tokenizer.pad_token_id
    lora_cfg = LoraConfig(task_type=TaskType.SEQ_CLS, r=lora_rank,
                          lora_alpha=lora_alpha, lora_dropout=lora_dropout,
                          target_modules=["c_attn"])
    model = get_peft_model(model, lora_cfg)
    model.print_trainable_parameters()
    return model, tokenizer


def build_t5(lora_rank, lora_alpha, lora_dropout):
    tokenizer = AutoTokenizer.from_pretrained("t5-small")
    model = T5ForConditionalGeneration.from_pretrained("t5-small")
    lora_cfg = LoraConfig(task_type=TaskType.SEQ_2_SEQ_LM, r=lora_rank,
                          lora_alpha=lora_alpha, lora_dropout=lora_dropout,
                          target_modules=["q", "v"])
    model = get_peft_model(model, lora_cfg)
    model.print_trainable_parameters()
    return model, tokenizer

## 5. Training Functions

In [ ]:
def train_cls_epoch(model, loader, optimizer, scheduler, label_smoothing=0.0,
                    grad_accum_steps=1, scaler=None):
    model.train()
    total_loss, all_preds, all_labels = 0, [], []
    loss_fn = nn.CrossEntropyLoss(label_smoothing=label_smoothing)
    use_amp = scaler is not None
    optimizer.zero_grad()

    for step, batch in enumerate(tqdm(loader, desc="  train", leave=False)):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        if use_amp:
            with torch.amp.autocast("cuda"):
                logits = model(input_ids=input_ids, attention_mask=attention_mask).logits
                loss = loss_fn(logits, labels) / grad_accum_steps
            scaler.scale(loss).backward()
        else:
            logits = model(input_ids=input_ids, attention_mask=attention_mask).logits
            loss = loss_fn(logits, labels) / grad_accum_steps
            loss.backward()

        total_loss += loss.item() * grad_accum_steps * input_ids.size(0)
        all_preds.extend(logits.argmax(-1).cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

        if (step + 1) % grad_accum_steps == 0 or (step + 1) == len(loader):
            if use_amp:
                scaler.unscale_(optimizer)
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(optimizer)
                scaler.update()
            else:
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
            scheduler.step()
            optimizer.zero_grad()

    n = len(loader.dataset)
    return total_loss / n, accuracy_score(all_labels, all_preds), f1_score(all_labels, all_preds, average="macro")


def eval_cls(model, loader, scaler=None):
    model.eval()
    total_loss, all_preds, all_labels = 0, [], []
    use_amp = scaler is not None
    with torch.no_grad():
        for batch in tqdm(loader, desc="  eval", leave=False):
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)
            if use_amp:
                with torch.amp.autocast("cuda"):
                    out = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            else:
                out = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            total_loss += out.loss.item() * input_ids.size(0)
            all_preds.extend(out.logits.argmax(-1).cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    n = len(loader.dataset)
    return total_loss / n, accuracy_score(all_labels, all_preds), f1_score(all_labels, all_preds, average="macro")


def t5_decode(tokenizer, ids):
    return tokenizer.decode(ids, skip_special_tokens=True).strip().lower()

NAME_TO_LABEL = {v: k for k, v in LABEL_NAMES.items()}

def t5_pred_to_label(text):
    c = text.strip().lower()
    if c in NAME_TO_LABEL: return NAME_TO_LABEL[c]
    for name, lab in NAME_TO_LABEL.items():
        if name in c: return lab
    return -1


def train_t5_epoch(model, loader, optimizer, scheduler, tokenizer):
    model.train()
    total_loss = 0
    for batch in tqdm(loader, desc="  train", leave=False):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)
        out = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        out.loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step(); scheduler.step(); optimizer.zero_grad()
        total_loss += out.loss.item() * input_ids.size(0)
    return total_loss / len(loader.dataset)


def eval_t5(model, loader, tokenizer):
    model.eval()
    total_loss, all_preds, all_labels = 0, [], []
    with torch.no_grad():
        for batch in tqdm(loader, desc="  eval", leave=False):
            ids = batch["input_ids"].to(device)
            mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)
            out = model(input_ids=ids, attention_mask=mask, labels=labels)
            total_loss += out.loss.item() * ids.size(0)
            gen = model.generate(input_ids=ids, attention_mask=mask, max_new_tokens=8)
            for g, tl in zip(gen, batch["genre_label"]):
                all_preds.append(t5_pred_to_label(t5_decode(tokenizer, g)))
                all_labels.append(tl.item())
    n = len(loader.dataset)
    pa, la = np.array(all_preds), np.array(all_labels)
    valid = pa >= 0
    acc = (pa[valid] == la[valid]).mean() if valid.sum() > 0 else 0.0
    f1 = f1_score(la[valid], pa[valid], average="macro", zero_division=0) if valid.sum() > 0 else 0.0
    return total_loss / n, acc, f1

## 6. Unified Training Loop

In [ ]:
def train_model(model_key, cfg=None, ckpt_dir=None, data_dir=None):
    if cfg is None: cfg = CFG
    if ckpt_dir is None: ckpt_dir = CKPT_DIR
    if data_dir is None: data_dir = DATA_DIR

    tc = cfg["training"]
    lc = cfg["lora"]
    is_t5 = model_key == "t5"
    grad_accum = tc.get("gradient_accumulation_steps", 1)
    use_fp16 = tc.get("fp16", False) and device.type == "cuda"
    augment = tc.get("augment", False)
    label_smoothing = tc.get("label_smoothing", 0.0)

    print(f"\n{'='*60}")
    print(f"  Training: {model_key.upper()}")
    print(f"  lr={tc['learning_rate']}  wd={tc['weight_decay']}  ls={label_smoothing}")
    print(f"  bs={tc['batch_size']}  accum={grad_accum}  fp16={use_fp16}  augment={augment}")
    print(f"  LoRA: r={lc['rank']}  alpha={lc['alpha']}  dropout={lc['dropout']}")
    print(f"  data_dir={data_dir}  ckpt_dir={ckpt_dir}")
    print(f"{'='*60}")

    if model_key == "roberta":
        target_mods = lc.get("target_modules_encoder", ["query", "value"])
        model, tokenizer = build_roberta(lc["rank"], lc["alpha"], lc["dropout"], target_mods)
    elif model_key == "gpt2":
        model, tokenizer = build_gpt2(lc["rank"], lc["alpha"], lc["dropout"])
    else:
        model, tokenizer = build_t5(lc["rank"], lc["alpha"], lc["dropout"])
    model = model.to(device)

    if is_t5:
        train_ds = T5LyricsDataset("train", tokenizer, tc["max_seq_len"], data_dir=data_dir)
        val_ds = T5LyricsDataset("validation", tokenizer, tc["max_seq_len"], data_dir=data_dir)
    else:
        train_ds = LyricsDataset("train", tokenizer, tc["max_seq_len"], data_dir=data_dir, augment=augment)
        val_ds = LyricsDataset("validation", tokenizer, tc["max_seq_len"], data_dir=data_dir)

    train_loader = DataLoader(train_ds, batch_size=tc["batch_size"], shuffle=True, num_workers=0)
    val_loader = DataLoader(val_ds, batch_size=tc["batch_size"], shuffle=False, num_workers=0)

    optimizer = torch.optim.AdamW(model.parameters(), lr=tc["learning_rate"], weight_decay=tc["weight_decay"])
    total_steps = (len(train_loader) // grad_accum) * tc["epochs"]
    warmup = int(total_steps * tc["warmup_ratio"])
    scheduler = get_cosine_schedule_with_warmup(optimizer, warmup, total_steps)

    scaler = None
    if use_fp16:
        scaler = torch.amp.GradScaler("cuda")
        print("  Mixed-precision (fp16) enabled")

    out_dir = Path(ckpt_dir) / model_key
    out_dir.mkdir(parents=True, exist_ok=True)
    best_f1, patience_ctr = 0.0, 0
    history = {"train_loss": [], "val_loss": [], "train_f1": [], "val_f1": [], "train_acc": [], "val_acc": []}

    for epoch in range(1, tc["epochs"] + 1):
        print(f"\n--- Epoch {epoch}/{tc['epochs']} ---")
        if is_t5:
            train_loss = train_t5_epoch(model, train_loader, optimizer, scheduler, tokenizer)
            val_loss, val_acc, val_f1 = eval_t5(model, val_loader, tokenizer)
            _, train_acc, train_f1 = eval_t5(model, train_loader, tokenizer)
        else:
            train_loss, train_acc, train_f1 = train_cls_epoch(
                model, train_loader, optimizer, scheduler,
                label_smoothing=label_smoothing, grad_accum_steps=grad_accum, scaler=scaler)
            val_loss, val_acc, val_f1 = eval_cls(model, val_loader, scaler)

        for k, v in [("train_loss", train_loss), ("val_loss", val_loss), ("train_f1", train_f1),
                     ("val_f1", val_f1), ("train_acc", train_acc), ("val_acc", val_acc)]:
            history[k].append(v)

        print(f"  Train -- loss: {train_loss:.4f}  acc: {train_acc:.4f}  f1: {train_f1:.4f}")
        print(f"  Val   -- loss: {val_loss:.4f}  acc: {val_acc:.4f}  f1: {val_f1:.4f}")

        if val_f1 > best_f1:
            best_f1, patience_ctr = val_f1, 0
            model.save_pretrained(str(out_dir / "best"))
            tokenizer.save_pretrained(str(out_dir / "best"))
            print(f"  >>> Saved best model (val_f1={best_f1:.4f})")
        else:
            patience_ctr += 1
            if patience_ctr >= tc["early_stopping_patience"]:
                print(f"  Early stopping after {epoch} epochs"); break

    with open(out_dir / "history.json", "w") as f:
        json.dump(history, f, indent=2)
    print(f"Best val F1: {best_f1:.4f}")

    del model, optimizer
    torch.cuda.empty_cache()
    return history

## 7. Phase 1: Train All Models

In [ ]:
hist_roberta = train_model("roberta")

In [ ]:
hist_gpt2 = train_model("gpt2")

In [ ]:
hist_t5 = train_model("t5")

## 8. Phase 1: Test-Set Evaluation

In [ ]:
def load_trained(model_key, ckpt_dir=CKPT_DIR):
    ckpt = str(Path(ckpt_dir) / model_key / "best")
    if model_key == "roberta":
        base = AutoModelForSequenceClassification.from_pretrained("roberta-base", num_labels=NUM_LABELS)
        model = PeftModel.from_pretrained(base, ckpt)
    elif model_key == "gpt2":
        base = GPT2ForSequenceClassification.from_pretrained("gpt2", num_labels=NUM_LABELS)
        base.config.pad_token_id = base.config.eos_token_id
        model = PeftModel.from_pretrained(base, ckpt)
    else:
        base = T5ForConditionalGeneration.from_pretrained("t5-small")
        model = PeftModel.from_pretrained(base, ckpt)
    tokenizer = AutoTokenizer.from_pretrained(ckpt)
    return model.to(device).eval(), tokenizer


GENRE_NAMES_DISP = ["Rap/Hip-Hop", "Pop", "Rock", "Country", "R&B"]
all_results = {}

for mk in ["roberta", "gpt2", "t5"]:
    print(f"\n{'='*50}\nEvaluating: {mk}\n{'='*50}")
    model, tokenizer = load_trained(mk)
    is_t5 = mk == "t5"

    if is_t5:
        test_ds = T5LyricsDataset("test", tokenizer, CFG["training"]["max_seq_len"])
    else:
        test_ds = LyricsDataset("test", tokenizer, CFG["training"]["max_seq_len"])
    test_loader = DataLoader(test_ds, batch_size=32, shuffle=False, num_workers=0)

    if is_t5:
        all_preds, all_labels = [], []
        with torch.no_grad():
            for batch in tqdm(test_loader, desc="Testing T5"):
                ids = batch["input_ids"].to(device)
                mask = batch["attention_mask"].to(device)
                gen = model.generate(input_ids=ids, attention_mask=mask, max_new_tokens=8)
                for g, tl in zip(gen, batch["genre_label"]):
                    all_preds.append(t5_pred_to_label(t5_decode(tokenizer, g)))
                    all_labels.append(tl.item())
        preds, labels = np.array(all_preds), np.array(all_labels)
        valid = preds >= 0
        preds, labels = preds[valid], labels[valid]
    else:
        all_preds, all_labels = [], []
        with torch.no_grad():
            for batch in tqdm(test_loader, desc=f"Testing {mk}"):
                batch = {k: v.to(device) for k, v in batch.items()}
                out = model(input_ids=batch["input_ids"], attention_mask=batch["attention_mask"])
                all_preds.extend(out.logits.argmax(-1).cpu().numpy())
                all_labels.extend(batch["labels"].cpu().numpy())
        preds, labels = np.array(all_preds), np.array(all_labels)

    acc = accuracy_score(labels, preds)
    f1m = f1_score(labels, preds, average="macro", zero_division=0)
    f1w = f1_score(labels, preds, average="weighted", zero_division=0)
    print(f"  Accuracy: {acc:.4f}  F1 macro: {f1m:.4f}  F1 weighted: {f1w:.4f}")
    print(classification_report(labels, preds, target_names=GENRE_NAMES_DISP, digits=4, zero_division=0))

    cm = confusion_matrix(labels, preds, labels=list(range(5)))
    plt.figure(figsize=(7, 6))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=GENRE_NAMES_DISP, yticklabels=GENRE_NAMES_DISP)
    plt.xlabel("Predicted"); plt.ylabel("True")
    plt.title(f"Confusion Matrix -- {mk.upper()}")
    plt.tight_layout()
    plt.savefig(f"{FIG_DIR}/cm_{mk}.png", dpi=150, bbox_inches="tight")
    plt.show()

    all_results[mk] = {"accuracy": float(acc), "f1_macro": float(f1m), "f1_weighted": float(f1w)}
    del model; torch.cuda.empty_cache()

# Comparison bar chart
model_names = list(all_results.keys())
metrics = ["accuracy", "f1_macro", "f1_weighted"]
x = np.arange(len(model_names))
width = 0.25
fig, ax = plt.subplots(figsize=(10, 5))
for i, metric in enumerate(metrics):
    vals = [all_results[m][metric] for m in model_names]
    bars = ax.bar(x + i * width, vals, width, label=metric.replace("_", " ").title())
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.005,
                f"{val:.3f}", ha="center", va="bottom", fontsize=9)
ax.set_ylabel("Score"); ax.set_title("Model Comparison on Test Set")
ax.set_xticks(x + width); ax.set_xticklabels([n.upper() for n in model_names])
ax.legend(); ax.set_ylim(0, 1.05)
plt.tight_layout()
plt.savefig(f"{FIG_DIR}/model_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

with open(f"{FIG_DIR}/test_results.json", "w") as f:
    json.dump(all_results, f, indent=2)
print("Phase 1 evaluation complete.")

---
## Phase 2: Optimized RoBERTa + Interpretability

Train RoBERTa with tuned hyperparameters on a larger dataset (5000/genre):
- LoRA rank 32 (alpha 64), targeting query/key/value
- LR 2e-5, gradient accumulation 4 (effective batch 128), fp16
- Label smoothing 0.05, random crop augmentation

In [ ]:
# Prepare Phase 2 dataset (5000 per genre)
CFG_OPTIMIZED = {
    "data": {
        "dataset_name": "sebastiandizon/genius-song-lyrics",
        "genres": ["rap", "pop", "rock", "country", "rb"],
        "samples_per_genre": 5000,
        "samples_per_genre_override": {"pop": 6000},
        "max_chars": 2000,
        "train_ratio": 0.8,
        "val_ratio": 0.1,
        "test_ratio": 0.1,
        "seed": 42,
    },
    "training": {
        "max_seq_len": 512,
        "batch_size": 32,
        "learning_rate": 1e-4,
        "weight_decay": 0.01,
        "epochs": 15,
        "warmup_ratio": 0.06,
        "early_stopping_patience": 4,
        "gradient_accumulation_steps": 2,
        "fp16": True,
        "label_smoothing": 0.0,
        "augment": True,
    },
    "lora": {
        "rank": 16,
        "alpha": 32,
        "dropout": 0.1,
        "target_modules_encoder": ["query", "key", "value"],
    },
}

DATA_DIR_P2 = "data/processed_phase2"
CKPT_DIR_OPT = "results/checkpoints_optimized"
FIG_DIR_OPT = "results/figures_optimized"
for d in [DATA_DIR_P2, CKPT_DIR_OPT, FIG_DIR_OPT,
          f"{FIG_DIR_OPT}/attention", f"{FIG_DIR_OPT}/attribution"]:
    os.makedirs(d, exist_ok=True)

dataset_p2 = prepare_dataset(CFG_OPTIMIZED["data"], LABEL_MAP, DATA_DIR_P2)

In [ ]:
# Train optimized RoBERTa
hist_opt = train_model("roberta", cfg=CFG_OPTIMIZED, ckpt_dir=CKPT_DIR_OPT, data_dir=DATA_DIR_P2)
print("Phase 2 RoBERTa saved to", CKPT_DIR_OPT)

In [ ]:
# Evaluate optimized RoBERTa on BOTH test sets for fair comparison
model, tokenizer = load_trained("roberta", ckpt_dir=CKPT_DIR_OPT)

def evaluate_on(model, tokenizer, data_dir, label, max_seq_len):
    ds = LyricsDataset("test", tokenizer, max_seq_len, data_dir=data_dir)
    loader = DataLoader(ds, batch_size=32, shuffle=False, num_workers=0)
    all_preds, all_labels = [], []
    with torch.no_grad():
        for batch in tqdm(loader, desc=f"Eval {label}"):
            batch = {k: v.to(device) for k, v in batch.items()}
            out = model(input_ids=batch["input_ids"], attention_mask=batch["attention_mask"])
            all_preds.extend(out.logits.argmax(-1).cpu().numpy())
            all_labels.extend(batch["labels"].cpu().numpy())
    preds, labels = np.array(all_preds), np.array(all_labels)
    acc = accuracy_score(labels, preds)
    f1m = f1_score(labels, preds, average="macro", zero_division=0)
    f1w = f1_score(labels, preds, average="weighted", zero_division=0)
    print(f"\n{label} -- Acc: {acc:.4f}  F1 macro: {f1m:.4f}  F1 weighted: {f1w:.4f}")
    print(classification_report(labels, preds, target_names=GENRE_NAMES_DISP, digits=4, zero_division=0))
    return preds, labels, {"accuracy": float(acc), "f1_macro": float(f1m), "f1_weighted": float(f1w)}

max_sl = CFG_OPTIMIZED["training"]["max_seq_len"]

# Evaluate on Phase 2 test set
preds_p2, labels_p2, res_p2 = evaluate_on(model, tokenizer, DATA_DIR_P2, "Optimized RoBERTa (Phase 2 test)", max_sl)

# Evaluate on Phase 1 test set (same test set as other models + Claude Opus)
preds_p1, labels_p1, res_p1 = evaluate_on(model, tokenizer, DATA_DIR, "Optimized RoBERTa (Phase 1 test)", max_sl)

# Plot confusion matrix for Phase 1 test (fair comparison)
cm = confusion_matrix(labels_p1, preds_p1, labels=list(range(5)))
plt.figure(figsize=(7, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=GENRE_NAMES_DISP, yticklabels=GENRE_NAMES_DISP)
plt.xlabel("Predicted"); plt.ylabel("True")
plt.title("Confusion Matrix -- RoBERTa (optimized, Phase 1 test)")
plt.tight_layout()
plt.savefig(f"{FIG_DIR_OPT}/cm_roberta_optimized.png", dpi=150, bbox_inches="tight")
plt.show()

opt_results = {
    "roberta_optimized_phase2_test": res_p2,
    "roberta_optimized_phase1_test": res_p1,
}
with open(f"{FIG_DIR_OPT}/test_results.json", "w") as f:
    json.dump(opt_results, f, indent=2)
del model; torch.cuda.empty_cache()

## 9. Embedding Visualization -- Optimized RoBERTa

In [ ]:
def extract_embeddings(model_key, ckpt_dir=CKPT_DIR_OPT, data_dir=DATA_DIR_P2):
    ckpt = str(Path(ckpt_dir) / model_key / "best")
    tokenizer = AutoTokenizer.from_pretrained(ckpt)
    base = AutoModelForSequenceClassification.from_pretrained(
        "roberta-base", num_labels=NUM_LABELS, output_hidden_states=True)
    model = PeftModel.from_pretrained(base, ckpt).to(device).eval()
    ds = LyricsDataset("test", tokenizer, CFG_OPTIMIZED["training"]["max_seq_len"], data_dir=data_dir)
    loader = DataLoader(ds, batch_size=32, shuffle=False)
    embs, labs = [], []
    with torch.no_grad():
        for batch in tqdm(loader, desc=f"{model_key} embeddings"):
            ids = batch["input_ids"].to(device)
            mask = batch["attention_mask"].to(device)
            out = model(input_ids=ids, attention_mask=mask, output_hidden_states=True)
            embs.append(out.hidden_states[-1][:, 0, :].cpu().numpy())
            labs.extend(batch["labels"].numpy())
    del model; torch.cuda.empty_cache()
    return np.concatenate(embs), np.array(labs)


def plot_2d(embs_2d, labels, model_key, method, out_dir):
    palette = sns.color_palette("Set2", n_colors=5)
    fig, ax = plt.subplots(figsize=(8, 7))
    for lid in sorted(set(labels)):
        m = labels == lid
        ax.scatter(embs_2d[m, 0], embs_2d[m, 1], label=GENRE_DISPLAY[lid],
                   color=palette[lid], alpha=0.7, s=30)
    ax.set_title(f"{model_key.upper()} -- {method} of Final-Layer Embeddings")
    ax.legend(title="Genre", loc="best")
    ax.set_xlabel(f"{method}-1"); ax.set_ylabel(f"{method}-2")
    plt.tight_layout()
    plt.savefig(f"{out_dir}/{method.lower()}_{model_key}.png", dpi=150, bbox_inches="tight")
    plt.show()


embs, labs = extract_embeddings("roberta")
print(f"Embeddings shape: {embs.shape}")

tsne = TSNE(n_components=2, perplexity=30, random_state=42, n_iter=1000)
embs_tsne = tsne.fit_transform(embs)
plot_2d(embs_tsne, labs, "roberta", "t-SNE", FIG_DIR_OPT)

if HAS_UMAP:
    reducer = umap.UMAP(n_components=2, random_state=42, n_neighbors=15, min_dist=0.1)
    embs_umap = reducer.fit_transform(embs)
    plot_2d(embs_umap, labs, "roberta", "UMAP", FIG_DIR_OPT)

np.savez(f"{FIG_DIR_OPT}/embeddings_roberta.npz", embeddings=embs, labels=labs)

## 10. Contrastive Attribution + Misclassification Analysis

For the most confused genre pairs, we attribute the **difference** in logits
(logit_A - logit_B) to input tokens using Integrated Gradients. This reveals
which words push the model toward one genre over another.

In [ ]:
from captum.attr import LayerIntegratedGradients

class ContrastiveWrapper(nn.Module):
    """Returns logit_A - logit_B for contrastive attribution."""
    def __init__(self, model, class_a, class_b):
        super().__init__()
        self.model = model
        self.class_a = class_a
        self.class_b = class_b

    def forward(self, input_ids, attention_mask):
        logits = self.model(input_ids=input_ids, attention_mask=attention_mask).logits
        return (logits[:, self.class_a] - logits[:, self.class_b]).unsqueeze(-1)


def get_emb_layer(model):
    base = getattr(model, "base_model", model)
    return base.roberta.embeddings.word_embeddings


def contrastive_ig(model, emb_layer, tokenizer, text, class_a, class_b,
                   max_length=512, n_steps=30):
    wrapper = ContrastiveWrapper(model, class_a, class_b)
    lig = LayerIntegratedGradients(wrapper, emb_layer)
    enc = tokenizer(text, truncation=True, max_length=max_length,
                    return_tensors="pt", padding=False)
    ids = enc["input_ids"].to(device)
    mask = enc["attention_mask"].to(device)
    attr = lig.attribute(ids, additional_forward_args=(mask,),
                         n_steps=n_steps, return_convergence_delta=False)
    scores = attr.sum(dim=-1).squeeze(0).cpu().numpy()
    tokens = tokenizer.convert_ids_to_tokens(ids[0].cpu())
    sl = mask.sum().item()
    return tokens[:sl], scores[:sl]


# Load model and get predictions on test set
model, tokenizer = load_trained("roberta", ckpt_dir=CKPT_DIR_OPT)
emb_layer = get_emb_layer(model)
test_data = load_from_disk(DATA_DIR_P2)["test"]
test_ds = LyricsDataset("test", tokenizer, CFG_OPTIMIZED["training"]["max_seq_len"], data_dir=DATA_DIR_P2)
test_loader = DataLoader(test_ds, batch_size=32, shuffle=False, num_workers=0)

all_preds, all_labels = [], []
with torch.no_grad():
    for batch in tqdm(test_loader, desc="Predicting"):
        batch = {k: v.to(device) for k, v in batch.items()}
        logits = model(input_ids=batch["input_ids"], attention_mask=batch["attention_mask"]).logits
        all_preds.extend(logits.argmax(-1).cpu().numpy())
        all_labels.extend(batch["labels"].cpu().numpy())
preds, labels = np.array(all_preds), np.array(all_labels)

# Find top 3 confused genre pairs
cm = confusion_matrix(labels, preds, labels=list(range(5)))
cm_off = cm.copy(); np.fill_diagonal(cm_off, 0)
confused_pairs = []
flat = cm_off.flatten()
for _ in range(3):
    idx = flat.argmax()
    i, j = divmod(idx, 5)
    confused_pairs.append((i, j, int(flat[idx])))
    flat[idx] = 0

print("Top confused genre pairs (true -> predicted):")
for true_g, pred_g, count in confused_pairs:
    print(f"  {GENRE_DISPLAY[true_g]} -> {GENRE_DISPLAY[pred_g]}: {count} errors")

# For each confused pair: contrastive attribution + vocabulary
max_len = CFG_OPTIMIZED["training"]["max_seq_len"]
os.makedirs(f"{FIG_DIR_OPT}/contrastive", exist_ok=True)

for true_g, pred_g, count in confused_pairs:
    pair_name = f"{GENRE_DISPLAY[true_g]}_vs_{GENRE_DISPLAY[pred_g]}".replace("/", "-")
    print(f"\n{'='*50}")
    print(f"Contrastive: {GENRE_DISPLAY[true_g]} vs {GENRE_DISPLAY[pred_g]} ({count} errors)")

    # Misclassified examples
    misclass_idx = np.where((labels == true_g) & (preds == pred_g))[0]
    for ex_i, idx in enumerate(misclass_idx[:2]):
        row = test_data[int(idx)]
        tokens, scores = contrastive_ig(model, emb_layer, tokenizer, row["lyrics"],
                                        true_g, pred_g, max_len)
        abs_scores = np.abs(scores)
        top_idx = np.argsort(abs_scores)[::-1][:20]
        top_tokens = [tokens[i].replace('\u0120', '').replace('\u2581', ' ').strip() for i in top_idx]
        top_scores = scores[top_idx]
        colors = [sns.color_palette('Set2')[0] if s > 0 else sns.color_palette('Set2')[1] for s in top_scores]

        fig, ax = plt.subplots(figsize=(9, 6))
        ax.barh(range(len(top_tokens)), top_scores, color=colors, alpha=0.85)
        ax.set_yticks(range(len(top_tokens))); ax.set_yticklabels(top_tokens, fontsize=9)
        ax.invert_yaxis()
        ax.set_xlabel(f'\u2190 {GENRE_DISPLAY[pred_g]}    Attribution    {GENRE_DISPLAY[true_g]} \u2192')
        ax.set_title(f'Misclassified: True={GENRE_DISPLAY[true_g]}, Pred={GENRE_DISPLAY[pred_g]}')
        ax.axvline(0, color='black', linewidth=0.5)
        plt.tight_layout()
        plt.savefig(f'{FIG_DIR_OPT}/contrastive/misclass_{pair_name}_{ex_i}.png', dpi=150, bbox_inches='tight')
        plt.show()

    # Contrastive vocabulary from both genres
    combined_rows = [r for r in test_data if r['label'] in (true_g, pred_g)]
    rng = np.random.default_rng(42)
    sample_idx = rng.choice(len(combined_rows), size=min(30, len(combined_rows)), replace=False)
    vocab_a, vocab_b = Counter(), Counter()
    for si in tqdm(sample_idx, desc=f'  Building vocab', leave=False):
        tokens, scores = contrastive_ig(model, emb_layer, tokenizer,
                                        combined_rows[int(si)]['lyrics'], true_g, pred_g, max_len, n_steps=20)
        for tok, s in zip(tokens, scores):
            clean = tok.replace('\u0120', '').replace('\u2581', '').lower().strip()
            if len(clean) > 2 and clean.isalpha():
                if s > 0: vocab_a[clean] += float(s)
                else: vocab_b[clean] += abs(float(s))

    top_a, top_b = vocab_a.most_common(20), vocab_b.most_common(20)
    if top_a and top_b:
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 7))
        wa, sa = zip(*top_a)
        ax1.barh(range(len(wa)), sa, color=sns.color_palette('Set2')[0], alpha=0.85)
        ax1.set_yticks(range(len(wa))); ax1.set_yticklabels(wa, fontsize=9)
        ax1.invert_yaxis(); ax1.set_xlabel('Cumulative Attribution')
        ax1.set_title(f'Tokens pushing toward {GENRE_DISPLAY[true_g]}')
        wb, sb = zip(*top_b)
        ax2.barh(range(len(wb)), sb, color=sns.color_palette('Set2')[1], alpha=0.85)
        ax2.set_yticks(range(len(wb))); ax2.set_yticklabels(wb, fontsize=9)
        ax2.invert_yaxis(); ax2.set_xlabel('Cumulative Attribution')
        ax2.set_title(f'Tokens pushing toward {GENRE_DISPLAY[pred_g]}')
        fig.suptitle(f'Contrastive Vocabulary: {GENRE_DISPLAY[true_g]} vs {GENRE_DISPLAY[pred_g]}', fontsize=13)
        plt.tight_layout()
        plt.savefig(f'{FIG_DIR_OPT}/contrastive/vocab_{pair_name}.png', dpi=150, bbox_inches='tight')
        plt.show()

# Normalized confusion matrix
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=GENRE_NAMES_DISP, yticklabels=GENRE_NAMES_DISP, ax=ax1)
ax1.set_xlabel('Predicted'); ax1.set_ylabel('True'); ax1.set_title('Confusion Matrix (counts)')
sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Oranges',
            xticklabels=GENRE_NAMES_DISP, yticklabels=GENRE_NAMES_DISP, ax=ax2)
ax2.set_xlabel('Predicted'); ax2.set_ylabel('True'); ax2.set_title('Confusion Matrix (row-normalized)')
plt.tight_layout()
plt.savefig(f'{FIG_DIR_OPT}/contrastive/confusion_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

del model; torch.cuda.empty_cache()
print('\nContrastive analysis complete.')

## 11. Layer-Wise Probing

Train a simple linear classifier on each RoBERTa layer's CLS representation.
This reveals **where** in the network genre information emerges:
- Early layers capture surface-level features (word identity, syntax)
- Later layers capture semantics
- A sharp rise at a specific layer shows where genre discrimination happens

In [ ]:
from sklearn.linear_model import LogisticRegression

def extract_all_layers(model, tokenizer, data_dir, split, max_seq_len, batch_size=32):
    ds = LyricsDataset(split, tokenizer, max_seq_len, data_dir=data_dir)
    loader = DataLoader(ds, batch_size=batch_size, shuffle=False, num_workers=0)
    layer_embs = {}
    all_labels = []
    with torch.no_grad():
        for batch in tqdm(loader, desc=f'Extracting ({split})'):
            ids = batch['input_ids'].to(device)
            mask = batch['attention_mask'].to(device)
            out = model(input_ids=ids, attention_mask=mask, output_hidden_states=True)
            for li, hidden in enumerate(out.hidden_states):
                cls = hidden[:, 0, :].cpu().numpy()
                layer_embs.setdefault(li, []).append(cls)
            all_labels.extend(batch['labels'].numpy())
    labels = np.array(all_labels)
    return {li: (np.concatenate(arrs), labels) for li, arrs in layer_embs.items()}


model, tokenizer = load_trained('roberta', ckpt_dir=CKPT_DIR_OPT)
# Need output_hidden_states — set in config
for attr_path in ('config', 'base_model.config', 'base_model.model.config'):
    obj = model
    for part in attr_path.split('.'):
        obj = getattr(obj, part, None)
        if obj is None: break
    if obj is not None:
        obj.output_hidden_states = True

max_len = CFG_OPTIMIZED['training']['max_seq_len']
print('Extracting all-layer embeddings ...')
train_layers = extract_all_layers(model, tokenizer, DATA_DIR_P2, 'train', max_len)
test_layers = extract_all_layers(model, tokenizer, DATA_DIR_P2, 'test', max_len)
del model; torch.cuda.empty_cache()

n_layers = len(train_layers)
print(f'Found {n_layers} layers (0=embeddings, 1-12=transformer)')

# Train linear probes
probe_results = {}
for li in tqdm(range(n_layers), desc='Probing'):
    X_tr, y_tr = train_layers[li]
    X_te, y_te = test_layers[li]
    clf = LogisticRegression(max_iter=1000, C=1.0, solver='lbfgs',
                             multi_class='multinomial', random_state=42)
    clf.fit(X_tr, y_tr)
    probe_results[li] = {
        'train_acc': float(accuracy_score(y_tr, clf.predict(X_tr))),
        'test_acc': float(accuracy_score(y_te, clf.predict(X_te))),
        'train_f1': float(f1_score(y_tr, clf.predict(X_tr), average='macro')),
        'test_f1': float(f1_score(y_te, clf.predict(X_te), average='macro')),
        'per_class_f1': {
            GENRE_DISPLAY[i]: float(f1_score(y_te == i, clf.predict(X_te) == i))
            for i in range(5)
        },
    }

# Print summary
print(f"{'Layer':>5}  {'Train Acc':>10}  {'Test Acc':>10}  {'Train F1':>10}  {'Test F1':>10}")
for li in sorted(probe_results.keys()):
    r = probe_results[li]
    print(f"{li:>5}  {r['train_acc']:>10.4f}  {r['test_acc']:>10.4f}  "
          f"{r['train_f1']:>10.4f}  {r['test_f1']:>10.4f}")

best_layer = max(probe_results, key=lambda l: probe_results[l]['test_f1'])
print(f"\nBest layer: {best_layer} (test F1 = {probe_results[best_layer]['test_f1']:.4f})")

# Plot accuracy/F1 by layer
layers = sorted(probe_results.keys())
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
ax1.plot(layers, [probe_results[l]['train_acc'] for l in layers], 'o-', label='Train', color='steelblue', alpha=0.7)
ax1.plot(layers, [probe_results[l]['test_acc'] for l in layers], 's-', label='Test', color='coral', alpha=0.9, lw=2)
ax1.set_xlabel('Layer'); ax1.set_ylabel('Accuracy'); ax1.set_title('Probing Accuracy by Layer')
ax1.legend(); ax1.set_xticks(layers); ax1.grid(True, alpha=0.3)

ax2.plot(layers, [probe_results[l]['train_f1'] for l in layers], 'o-', label='Train', color='steelblue', alpha=0.7)
ax2.plot(layers, [probe_results[l]['test_f1'] for l in layers], 's-', label='Test', color='coral', alpha=0.9, lw=2)
ax2.set_xlabel('Layer'); ax2.set_ylabel('Macro F1'); ax2.set_title('Probing F1 by Layer')
ax2.legend(); ax2.set_xticks(layers); ax2.grid(True, alpha=0.3)
best_f1 = probe_results[best_layer]['test_f1']
ax2.annotate(f'Best: Layer {best_layer}\nF1={best_f1:.3f}',
             xy=(best_layer, best_f1), xytext=(best_layer - 2, best_f1 - 0.05),
             arrowprops=dict(arrowstyle='->', color='black'), fontsize=10)
plt.tight_layout()
plt.savefig(f'{FIG_DIR_OPT}/layer_probing.png', dpi=150, bbox_inches='tight')
plt.show()

# Per-genre F1 by layer
palette = sns.color_palette('Set2', n_colors=5)
fig, ax = plt.subplots(figsize=(10, 6))
for gid in range(5):
    f1s = [probe_results[l]['per_class_f1'][GENRE_DISPLAY[gid]] for l in layers]
    ax.plot(layers, f1s, 'o-', label=GENRE_DISPLAY[gid], color=palette[gid], lw=2, alpha=0.85)
ax.set_xlabel('Layer'); ax.set_ylabel('F1 Score'); ax.set_title('Per-Genre Probing F1 by Layer')
ax.legend(title='Genre', loc='lower right'); ax.set_xticks(layers); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(f'{FIG_DIR_OPT}/layer_probing_per_genre.png', dpi=150, bbox_inches='tight')
plt.show()

with open(f'{FIG_DIR_OPT}/layer_probing_results.json', 'w') as f:
    json.dump({str(k): v for k, v in probe_results.items()}, f, indent=2)
print('Layer probing complete.')

## 12. Download Results

Download the `results/` folder and data. Then run locally:
```bash
# Genre similarity analysis (no GPU needed)
python -m src.analysis.genre_similarity --data_dir data/processed

# Claude API baseline (no GPU needed, requires ANTHROPIC_API_KEY)
python -m src.analysis.llm_baseline --data_dir data/processed
```

In [ ]:
!zip -r results.zip results/ data/processed/ data/processed_phase2/
from google.colab import files
files.download("results.zip")